Restarted .venv-ml (Python 3.10.12)

In [ ]:
import numpy as np
from pathlib import Path
from scipy.sparse import csr_matrix, vstack
from sklearn.linear_model import SGDClassifier
# from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
N = None
batch_dir = Path("./data/sae_vectors")
batch_files = sorted(batch_dir.glob("confirmatory_preprocessed_sae_vectors_sparse_minibatch_*.npz"))
batch_files = batch_files[:N] if N is not None else batch_files

Xs = []
for f in batch_files:
    z = np.load(f)
    Xs.append(csr_matrix((z["data"], z["indices"], z["indptr"]), shape=tuple(z["shape"])))

X = vstack(Xs, format="csr")  # rows = examples
# print(X.shape, X.nnz)

best_files = sorted(batch_dir.glob("confirmatory_preprocessed_sae_vectors_best_minibatch_*.npy"))
best_files = best_files[:N] if N is not None else best_files
y = np.concatenate([np.load(f).reshape(-1) for f in best_files], axis=0)
# print(y.shape)

In [ ]:
# Each pair has 2 rows: row 2i = best-worst, row 2i+1 = worst-best
# We must keep pairs together since worst-best = -1 * (best-worst)
print("Splitting data into train and test sets (by pair)...")
n_pairs = X.shape[0] // 2
pair_indices = np.arange(n_pairs)

# Split at the pair level
train_pairs, test_pairs = train_test_split(pair_indices, test_size=0.2, random_state=42)

# Convert pair indices to row indices
train_idx = np.sort(np.concatenate([train_pairs * 2, train_pairs * 2 + 1]))
test_idx = np.sort(np.concatenate([test_pairs * 2, test_pairs * 2 + 1]))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {len(train_pairs)} pairs ({X_train.shape[0]} rows), Test: {len(test_pairs)} pairs ({X_test.shape[0]} rows)")

Splitting data into train and test sets (by pair)...
Train: 10777 pairs (21554 rows), Test: 2695 pairs (5390 rows)


In [ ]:
scaler = StandardScaler(with_mean=False)
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
print("Training logistic regression model...")

model = SGDClassifier(loss='log_loss', penalty='l2', alpha=0.0001, max_iter=2000, tol=1e-3, verbose=1, n_jobs=-1, random_state=0)

model.fit(X_train, y_train)

Training logistic regression model...
-- Epoch 1
Norm: 2767.74, NNZs: 13683, Bias: 0.010649, T: 21554, Avg. loss: 1023.188044
Total training time: 0.03 seconds.
-- Epoch 2
Norm: 1624.36, NNZs: 13769, Bias: -0.192873, T: 43108, Avg. loss: 76.088521
Total training time: 0.06 seconds.
-- Epoch 3
Norm: 1170.13, NNZs: 13806, Bias: -0.074226, T: 64662, Avg. loss: 17.672273
Total training time: 0.08 seconds.
-- Epoch 4
Norm: 910.52, NNZs: 13821, Bias: -0.045259, T: 86216, Avg. loss: 5.605412
Total training time: 0.11 seconds.
-- Epoch 5
Norm: 745.21, NNZs: 13828, Bias: -0.047157, T: 107770, Avg. loss: 3.114189
Total training time: 0.14 seconds.
-- Epoch 6
Norm: 632.42, NNZs: 13835, Bias: -0.044311, T: 129324, Avg. loss: 1.921013
Total training time: 0.17 seconds.
-- Epoch 7
Norm: 551.35, NNZs: 13843, Bias: -0.039920, T: 150878, Avg. loss: 1.793277
Total training time: 0.20 seconds.
-- Epoch 8
Norm: 486.95, NNZs: 13847, Bias: -0.035677, T: 172432, Avg. loss: 1.096670
Total training time: 0.24 

,loss,'log_loss'
,penalty,'l2'
,alpha,0.0001
,l1_ratio,0.15
,fit_intercept,True
,max_iter,2000
,tol,0.001
,shuffle,True
,verbose,1
,epsilon,0.1
,n_jobs,-1


In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print(classification_report(y_test, y_pred))
# print(confusion_matrix(y_test, y_pred))

Accuracy: 0.6808905380333952
              precision    recall  f1-score   support

           0       0.67      0.70      0.69      2695
           1       0.69      0.66      0.67      2695

    accuracy                           0.68      5390
   macro avg       0.68      0.68      0.68      5390
weighted avg       0.68      0.68      0.68      5390



In [ ]:
feature_importances = model.coef_
# feature_importances = model.feature_importances_.reshape(1, -1)
# print(feature_importances.shape)
top10 = np.argsort(feature_importances.flatten())[-10:][::-1]
bottom10 = np.argsort(feature_importances.flatten())[:10]
print("\nTop 10 feature indices and importance scores:")
for idx in top10:
    print(f"Feature {idx}: {feature_importances[0][idx]}")
print("\nBottom 10 feature indices and importance scores:")
for idx in bottom10:
    print(f"Feature {idx}: {feature_importances[0][idx]}")


Top 10 feature indices and importance scores:
Feature 9866: 3.126314401626587
Feature 9100: 2.6805121898651123
Feature 5439: 2.6666007041931152
Feature 6143: 2.565145492553711
Feature 9859: 2.5480878353118896
Feature 2824: 2.4787280559539795
Feature 13844: 2.3822505474090576
Feature 3202: 2.2764394283294678
Feature 3759: 2.225837469100952
Feature 11052: 2.2197012901306152

Bottom 10 feature indices and importance scores:
Feature 15081: -3.9798941612243652
Feature 11541: -2.556295156478882
Feature 3638: -2.39022159576416
Feature 6497: -2.357759475708008
Feature 2109: -2.3524563312530518
Feature 11106: -2.3206305503845215
Feature 8276: -2.3161604404449463
Feature 4776: -2.2398877143859863
Feature 16071: -2.1350903511047363
Feature 6050: -2.125060796737671


In [ ]:
from IPython.display import IFrame
html_template = "https://neuronpedia.org/{}/{}/{}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"
# example_url = "https://neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-16k/1500?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

def get_dashboard_html(sae_release = "gemma-3-4b-it", sae_id="22-gemmascope-2-res-16k", feature_idx=0):
    return html_template.format(sae_release, sae_id, feature_idx)

html = get_dashboard_html(feature_idx=top10[0])
IFrame(html, width=1200, height=600)

In [ ]:
# class exact match: "text-left font-sans text-[11.5px] sm:text-[13px]"
# if it exists
import requests
from bs4 import BeautifulSoup
def get_explanation(url):
    response = requests.get(url)
    html_content = response.text
    soup = BeautifulSoup(html_content, 'html.parser')
    explanation = soup.find_all(class_="text-left font-sans text-[11.5px] sm:text-[13px]")
    if explanation:
        return explanation[0].text
    return None

print("\nTop 10 feature explanations:")
for feature_idx in top10:
    url = get_dashboard_html(feature_idx=feature_idx)
    explanation = get_explanation(url)
    print(f"Feature {feature_idx}: {explanation}")

print("\nBottom 10 feature explanations:")
for feature_idx in bottom10:
    url = get_dashboard_html(feature_idx=feature_idx)
    explanation = get_explanation(url)
    print(f"Feature {feature_idx}: {explanation}")



Top 10 feature explanations:
Feature 9866: mark correct answers
Feature 9100: the most proper one
Feature 5439: offensive content
Feature 6143: designed purpose
Feature 9859: objective factual rational
Feature 2824: corporate center and data models
Feature 13844: possessive pronouns (my, your, his)
Feature 3202: item characteristics
Feature 3759: evil AI, monsters
Feature 11052: How to write clickbait titles

Bottom 10 feature explanations:
Feature 15081: i understand you
Feature 11541: historical and natural landmarks
Feature 3638: historical and social contexts
Feature 6497: adverse health symptoms
Feature 2109: responsible for tasks
Feature 11106: Prep time
Feature 8276: pronouns followed by actions
Feature 4776: describing specific elements and events
Feature 16071: garbage, trash, and disposal
Feature 6050: guaranteed limits


In [ ]:
from sklearn.preprocessing import MaxAbsScaler

In [ ]:
# Each pair has 2 rows: row 2i = best-worst, row 2i+1 = worst-best
# We must keep pairs together since worst-best = -1 * (best-worst)
print("Splitting data into train and test sets (by pair)...")
n_pairs = X.shape[0] // 2
pair_indices = np.arange(n_pairs)

# Split at the pair level
train_pairs, test_pairs = train_test_split(pair_indices, test_size=0.2, random_state=42)

# Convert pair indices to row indices
train_idx = np.sort(np.concatenate([train_pairs * 2, train_pairs * 2 + 1]))
test_idx = np.sort(np.concatenate([test_pairs * 2, test_pairs * 2 + 1]))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {len(train_pairs)} pairs ({X_train.shape[0]} rows), Test: {len(test_pairs)} pairs ({X_test.shape[0]} rows)")

Splitting data into train and test sets (by pair)...
Train: 10777 pairs (21554 rows), Test: 2695 pairs (5390 rows)


In [ ]:
# scaler = StandardScaler(with_mean=False)
scaler = MaxAbsScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
print("Training logistic regression model...")

model = SGDClassifier(loss='log_loss', penalty='l2', alpha=0.0001, max_iter=2000, tol=1e-3, verbose=1, n_jobs=-1, random_state=0)

model.fit(X_train, y_train)

Training logistic regression model...
-- Epoch 1
Norm: 193.12, NNZs: 14133, Bias: -0.039454, T: 21554, Avg. loss: 9.942361
Total training time: 0.04 seconds.
-- Epoch 2
Norm: 121.83, NNZs: 14172, Bias: 0.052145, T: 43108, Avg. loss: 1.105709
Total training time: 0.08 seconds.
-- Epoch 3
Norm: 92.13, NNZs: 14172, Bias: -0.035600, T: 64662, Avg. loss: 0.404846
Total training time: 0.12 seconds.
-- Epoch 4
Norm: 74.48, NNZs: 14172, Bias: 0.006845, T: 86216, Avg. loss: 0.196692
Total training time: 0.16 seconds.
-- Epoch 5
Norm: 63.20, NNZs: 14172, Bias: -0.012895, T: 107770, Avg. loss: 0.139837
Total training time: 0.20 seconds.
-- Epoch 6
Norm: 55.87, NNZs: 14172, Bias: -0.003293, T: 129324, Avg. loss: 0.124904
Total training time: 0.24 seconds.
-- Epoch 7
Norm: 51.07, NNZs: 14172, Bias: 0.001603, T: 150878, Avg. loss: 0.128270
Total training time: 0.27 seconds.
-- Epoch 8
Norm: 47.84, NNZs: 14172, Bias: -0.002412, T: 172432, Avg. loss: 0.130509
Total training time: 0.31 seconds.
-- Epoc

,loss,'log_loss'
,penalty,'l2'
,alpha,0.0001
,l1_ratio,0.15
,fit_intercept,True
,max_iter,2000
,tol,0.001
,shuffle,True
,verbose,1
,epsilon,0.1
,n_jobs,-1


In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print(classification_report(y_test, y_pred))
# print(confusion_matrix(y_test, y_pred))

Accuracy: 0.700556586270872
              precision    recall  f1-score   support

           0       0.71      0.68      0.69      2695
           1       0.69      0.72      0.71      2695

    accuracy                           0.70      5390
   macro avg       0.70      0.70      0.70      5390
weighted avg       0.70      0.70      0.70      5390



In [ ]:
feature_importances = model.coef_
# feature_importances = model.feature_importances_.reshape(1, -1)
# print(feature_importances.shape)
top10 = np.argsort(feature_importances.flatten())[-10:][::-1]
bottom10 = np.argsort(feature_importances.flatten())[:10]
print("\nTop 10 feature indices and importance scores:")
for idx in top10:
    print(f"Feature {idx}: {feature_importances[0][idx]}")
print("\nBottom 10 feature indices and importance scores:")
for idx in bottom10:
    print(f"Feature {idx}: {feature_importances[0][idx]}")


Top 10 feature indices and importance scores:
Feature 3759: 1.7220655679702759
Feature 5439: 1.716261386871338
Feature 424: 1.479891300201416
Feature 2895: 1.424490213394165
Feature 7331: 1.407021164894104
Feature 9859: 1.4050703048706055
Feature 7345: 1.3747516870498657
Feature 433: 1.330073356628418
Feature 7496: 1.3292042016983032
Feature 813: 1.321831226348877

Bottom 10 feature indices and importance scores:
Feature 8960: -1.654599666595459
Feature 5952: -1.4464327096939087
Feature 5662: -1.3955018520355225
Feature 5396: -1.3849049806594849
Feature 5951: -1.3794279098510742
Feature 9080: -1.3580610752105713
Feature 10015: -1.3337675333023071
Feature 6497: -1.327833890914917
Feature 9135: -1.30411958694458
Feature 2028: -1.300144910812378


In [ ]:
from IPython.display import IFrame
html_template = "https://neuronpedia.org/{}/{}/{}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"
# example_url = "https://neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-16k/1500?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

def get_dashboard_html(sae_release = "gemma-3-4b-it", sae_id="22-gemmascope-2-res-16k", feature_idx=0):
    return html_template.format(sae_release, sae_id, feature_idx)

html = get_dashboard_html(feature_idx=top10[0])
IFrame(html, width=1200, height=600)

In [ ]:
# class exact match: "text-left font-sans text-[11.5px] sm:text-[13px]"
# if it exists
import requests
from bs4 import BeautifulSoup
def get_explanation(url):
    response = requests.get(url)
    html_content = response.text
    soup = BeautifulSoup(html_content, 'html.parser')
    explanation = soup.find_all(class_="text-left font-sans text-[11.5px] sm:text-[13px]")
    if explanation:
        return explanation[0].text
    return None

print("\nTop 10 feature explanations:")
for feature_idx in top10:
    url = get_dashboard_html(feature_idx=feature_idx)
    explanation = get_explanation(url)
    print(f"Feature {feature_idx}: {explanation}")

print("\nBottom 10 feature explanations:")
for feature_idx in bottom10:
    url = get_dashboard_html(feature_idx=feature_idx)
    explanation = get_explanation(url)
    print(f"Feature {feature_idx}: {explanation}")


Top 10 feature explanations:
Feature 3759: evil AI, monsters
Feature 5439: offensive content
Feature 424: 
Feature 2895: suggest related
Feature 7331: dismissing negative perceptions
Feature 9859: objective factual rational
Feature 7345: qualifying observations
Feature 433: This neuron spots words and phrases that introduce or label problems—like “issue,” “breakdown,” “core problem,” or other signals that a difficulty is being explained.
Feature 7496: vaginal and penile terms
Feature 813: 

Bottom 10 feature explanations:
Feature 8960: if you or should you
Feature 5952: neighborly interactions
Feature 5662: descriptive and positive adjectives
Feature 5396: resources for further reading or learning
Feature 5951: dialogue quotations and first person
Feature 9080: Title slide or image description
Feature 10015: essential key concepts
Feature 6497: adverse health symptoms
Feature 9135: The Gemma model
Feature 2028: social and group performance


In [ ]:
# %% import libraries

import numpy as np
from pathlib import Path
from scipy.sparse import csr_matrix, vstack
from sklearn.linear_model import SGDClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
# %% load data

N = None
batch_dir = Path("./data/sae_vectors")
batch_files = sorted(batch_dir.glob("confirmatory_preprocessed_sae_vectors_sparse_minibatch_*.npz"))
batch_files = batch_files[:N] if N is not None else batch_files

Xs = []
for f in batch_files:
    z = np.load(f)
    Xs.append(csr_matrix((z["data"], z["indices"], z["indptr"]), shape=tuple(z["shape"])))

X = vstack(Xs, format="csr")  # rows = examples
# print(X.shape, X.nnz)

best_files = sorted(batch_dir.glob("confirmatory_preprocessed_sae_vectors_best_minibatch_*.npy"))
best_files = best_files[:N] if N is not None else best_files
y = np.concatenate([np.load(f).reshape(-1) for f in best_files], axis=0)
# print(y.shape)

In [ ]:
# %% split data (by pairs to avoid data leakage)

# Each pair has 2 rows: row 2i = best-worst, row 2i+1 = worst-best
# We must keep pairs together since worst-best = -1 * (best-worst)
print("Splitting data into train and test sets (by pair)...")
n_pairs = X.shape[0] // 2
pair_indices = np.arange(n_pairs)

# Split at the pair level
train_pairs, test_pairs = train_test_split(pair_indices, test_size=0.2, random_state=42)

# Convert pair indices to row indices
train_idx = np.sort(np.concatenate([train_pairs * 2, train_pairs * 2 + 1]))
test_idx = np.sort(np.concatenate([test_pairs * 2, test_pairs * 2 + 1]))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {len(train_pairs)} pairs ({X_train.shape[0]} rows), Test: {len(test_pairs)} pairs ({X_test.shape[0]} rows)")

Splitting data into train and test sets (by pair)...
Train: 10777 pairs (21554 rows), Test: 2695 pairs (5390 rows)


In [ ]:
# %% standardize features

scaler = StandardScaler(with_mean=False)
# scaler = MaxAbsScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# %% train XGBoost model

model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    n_jobs=-1,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=1 
)

model.fit(X_train, y_train)

Training logistic regression model...


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [ ]:
# %% evaluate model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print(classification_report(y_test, y_pred))
# print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7092764378478664
              precision    recall  f1-score   support

           0       0.70      0.73      0.72      2695
           1       0.72      0.68      0.70      2695

    accuracy                           0.71      5390
   macro avg       0.71      0.71      0.71      5390
weighted avg       0.71      0.71      0.71      5390



In [ ]:
# %% feature importance

if isinstance(model, SGDClassifier):
    feature_importances = model.coef_
elif isinstance(model, XGBClassifier):
    feature_importances = model.feature_importances_.reshape(1, -1)
else:
    raise ValueError("Unsupported model type for feature importance extraction.")

top10 = np.argsort(feature_importances.flatten())[-10:][::-1]
bottom10 = np.argsort(feature_importances.flatten())[:10]
print("\nTop 10 feature indices and importance scores:")
for idx in top10:
    print(f"Feature {idx}: {feature_importances[0][idx]}")
print("\nBottom 10 feature indices and importance scores:")
for idx in bottom10:
    print(f"Feature {idx}: {feature_importances[0][idx]}")


Top 10 feature indices and importance scores:
Feature 1531: 0.0029667848721146584
Feature 11052: 0.0029070614837110043
Feature 1389: 0.002652724739164114
Feature 2727: 0.0025454084388911724
Feature 3669: 0.002351522445678711
Feature 538: 0.0023318335879594088
Feature 2400: 0.0018853547517210245
Feature 1213: 0.001845110789872706
Feature 444: 0.0018144473433494568
Feature 12139: 0.001767763402312994

Bottom 10 feature indices and importance scores:
Feature 10820: 0.0
Feature 10805: 0.0
Feature 10806: 0.0
Feature 10807: 0.0
Feature 10808: 0.0
Feature 10809: 0.0
Feature 10810: 0.0
Feature 10811: 0.0
Feature 10812: 0.0
Feature 10813: 0.0


In [ ]:
# %% get feature names from neuronpedia

from IPython.display import IFrame
html_template = "https://neuronpedia.org/{}/{}/{}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"
# example_url = "https://neuronpedia.org/gemma-3-4b-it/22-gemmascope-2-res-16k/1500?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

def get_dashboard_html(sae_release = "gemma-3-4b-it", sae_id="22-gemmascope-2-res-16k", feature_idx=0):
    return html_template.format(sae_release, sae_id, feature_idx)

html = get_dashboard_html(feature_idx=top10[0])
IFrame(html, width=1200, height=600)

In [ ]:
# %% get explanations

# class exact match: "text-left font-sans text-[11.5px] sm:text-[13px]"
# if it exists
import requests
from bs4 import BeautifulSoup
def get_explanation(url):
    response = requests.get(url)
    html_content = response.text
    soup = BeautifulSoup(html_content, 'html.parser')
    explanation = soup.find_all(class_="text-left font-sans text-[11.5px] sm:text-[13px]")
    if explanation:
        return explanation[0].text
    return None

print("\nTop 10 feature explanations:")
for feature_idx in top10:
    url = get_dashboard_html(feature_idx=feature_idx)
    explanation = get_explanation(url)
    print(f"Feature {feature_idx}: {explanation}")

print("\nBottom 10 feature explanations:")
for feature_idx in bottom10:
    url = get_dashboard_html(feature_idx=feature_idx)
    explanation = get_explanation(url)
    print(f"Feature {feature_idx}: {explanation}")


Top 10 feature explanations:
Feature 1531: discovery and revelation
Feature 11052: How to write clickbait titles
Feature 1389: 
Feature 2727: internet social media slang
Feature 3669: video content and analysis
Feature 538: restrict public trade abides necessitate
Feature 2400: a, say, have sequences
Feature 1213: 
Feature 444: 
Feature 12139: questions and technical topics

Bottom 10 feature explanations:
Feature 10820: commas followed by country names
Feature 10805: scene understanding
Feature 10806: family members and their actions
Feature 10807: bin and lib directories
Feature 10808: followed by superlatives or comparatives
Feature 10809: css white or #fff
Feature 10810: higher or increasing magnitude
Feature 10811: immediate actions and steps
Feature 10812: cyberpunk or cybersecurity
Feature 10813: people's names
